In [1]:
import glob
for f in ("preprocessor.joblib", "random_forest.joblib", "test_features.parquet",
          "metrics.py", "predict_window.py", "checkpoint.pth"):
    print(f, glob.glob(f"/kaggle/input/**/{f}", recursive=True))

preprocessor.joblib ['/kaggle/input/datasets/lkhangtannl/nids-phase1-output/processed/preprocessor.joblib']
random_forest.joblib ['/kaggle/input/datasets/lkhangtannl/nids-phase1-output/models/random_forest.joblib']
test_features.parquet ['/kaggle/input/datasets/lkhangtannl/nids-phase1-output/processed/test_features.parquet']
metrics.py ['/kaggle/input/datasets/lkhangtannl/nids-phase1-output/metrics.py']
predict_window.py ['/kaggle/input/datasets/lkhangtannl/nids-gnn-b/predict_window.py']
checkpoint.pth ['/kaggle/input/datasets/lkhangtannl/nids-gnn-b/checkpoint.pth']


### Cell 1: Cài thư viện, nạp dữ liệu (chỉ đọc)

In [2]:
%pip install -q torch_geometric
import glob, os, sys, time, queue, threading, warnings, joblib
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def find(name):
    h = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(h) == 1, (name, h)
    return h[0]

PRE, CKPT = find("preprocessor.joblib"), find("checkpoint.pth")
RF_PATH, TEST_X = find("random_forest.joblib"), find("test_features.parquet")
PROC = os.path.dirname(TEST_X)
for f in ("predict_window.py", "metrics.py"):
    sys.path.insert(0, os.path.dirname(find(f)))
import metrics

pre = joblib.load(PRE)
X = pd.read_parquet(TEST_X)
meta = pd.read_parquet(f"{PROC}/test_meta.parquet")
assert list(X.columns) == list(pre["feature_cols"]) and len(X) == len(meta)
order = np.argsort(meta["timestamp"].to_numpy(), kind="stable")   # thứ tự phát lại
y = meta["label_bin"].to_numpy()

# giả lập flow thô bằng cách hoàn nguyên chuẩn hóa (không sửa dữ liệu gốc)
raw = pd.DataFrame(pre["scaler"].inverse_transform(X.to_numpy()), columns=pre["feature_cols"])
raw["src_ip"], raw["dst_ip"] = meta["src_ip"].values, meta["dst_ip"].values
print(X.shape, "OK")

Note: you may need to restart the kernel to use updated packages.
(424591, 70) OK


### Cell 1.1

In [3]:
import torch
from predict_window import predict_window

from sklearn.metrics import f1_score

W = 5000
def probs_for(df_raw, o):
    pr = np.zeros(len(df_raw))
    for s in range(0, len(o), W):
        idx = o[s:s + W]
        pr[idx] = predict_window(df_raw.iloc[idx].reset_index(drop=True),
                                 checkpoint_path=CKPT, preprocessor_path=PRE)[1]
    return pr

# (a) ngưỡng tối ưu trên Val và Test với cửa sổ 5000
vx = glob.glob("/kaggle/input/**/val_features.parquet", recursive=True)[0]
Xv = pd.read_parquet(vx); mv = pd.read_parquet(vx.replace("val_features", "val_meta"))
rawv = pd.DataFrame(pre["scaler"].inverse_transform(Xv.to_numpy()), columns=pre["feature_cols"])
rawv["src_ip"], rawv["dst_ip"] = mv["src_ip"].values, mv["dst_ip"].values
ov = np.argsort(mv["timestamp"].to_numpy(), kind="stable")
pv, pt = probs_for(rawv, ov), probs_for(raw, order)
yv = mv["label_bin"].to_numpy()
ths = np.round(np.arange(0.05, 0.96, 0.05), 2)
for name, yy, pp in (("val", yv, pv), ("test", y, pt)):
    f = [f1_score(yy, (pp >= t).astype(int)) for t in ths]
    print(name, "| ngưỡng tốt nhất:", ths[int(np.argmax(f))], "| F1 tốt nhất:", round(max(f), 4),
          "| F1@0.15:", round(f1_score(yy, (pp >= 0.15).astype(int)), 4))

# (b) thứ tự flow trùng timestamp có làm lệch kết quả không
ts_arr = meta["timestamp"].to_numpy()
for name, o in (("argsort mặc định", np.argsort(ts_arr)),
                ("sort_values", meta.sort_values("timestamp").index.to_numpy()),
                ("stable (của mình)", order)):
    cm = metrics.compute_metrics(y, (probs_for(raw, o) >= 0.15).astype(int))["confusion_matrix"]
    print(f"{name:20s} FP={cm[0][1]}  FN={cm[1][0]}   (B: 960, 14749)")

val | ngưỡng tốt nhất: 0.8 | F1 tốt nhất: 0.9969 | F1@0.15: 0.9936
test | ngưỡng tốt nhất: 0.4 | F1 tốt nhất: 0.8974 | F1@0.15: 0.8937
argsort mặc định     FP=960  FN=14749   (B: 960, 14749)
sort_values          FP=960  FN=14749   (B: 960, 14749)
stable (của mình)    FP=930  FN=15327   (B: 960, 14749)


### Cell 2: Định nghĩa pipeline

In [4]:
import torch
OUT = "/kaggle/working/streaming_results"; os.makedirs(OUT, exist_ok=True)
THREADS = 1                      # đổi thành 4 rồi chạy lại nếu muốn so sánh
torch.set_num_threads(THREADS)
rf = joblib.load(RF_PATH); rf.set_params(n_jobs=THREADS)
FEATS, MED, SC = list(pre["feature_cols"]), pre["medians"], pre["scaler"]

def fn_rf(df):   # tiền xử lý đúng như lúc huấn luyện + dự đoán xác suất Attack
    Xw = df[FEATS].replace([np.inf, -np.inf], np.nan).fillna(MED)
    return rf.predict_proba(SC.transform(Xw).astype("float32"))[:, 1]
def fn_gnn(df):
    return predict_window(df, checkpoint_path=CKPT, preprocessor_path=PRE)[1]
FNS = {"rf": fn_rf, "gnn": fn_gnn}
for f_ in FNS.values(): f_(raw.iloc[order[:2000]].reset_index(drop=True))   # khởi động, không tính giờ

CH = 100
def run_stream(name, W, rate, idx_all):
    fn, q = FNS[name], queue.Queue()
    probs = np.full(len(raw), np.nan); wins, lats = [], []
    def producer():
        t_next = time.perf_counter()
        for s in range(0, len(idx_all), CH):
            if rate:
                t_next += CH / rate
                d = t_next - time.perf_counter()
                if d > 0: time.sleep(d)
            q.put((idx_all[s:s + CH], time.perf_counter()))
        q.put(None)
    def process(ids, tin, wid):
        ta = time.perf_counter()
        df = raw.iloc[ids].reset_index(drop=True)
        t0 = time.perf_counter(); p = fn(df); t1 = time.perf_counter()
        probs[ids] = p; l = t1 - tin; lats.append(l)
        wins.append({"window_id": wid, "n": len(ids), "assemble_s": t0 - ta, "predict_s": t1 - t0,
                     "lat_mean_s": l.mean(), "lat_max_s": l.max()})
    def consumer():
        ids, tin, n, wid = [], [], 0, 0
        while True:
            m = q.get()
            if m is None:
                if n: process(np.concatenate(ids), np.concatenate(tin), wid)
                return
            ids.append(m[0]); tin.append(np.full(len(m[0]), m[1])); n += len(m[0])
            if n >= W:
                process(np.concatenate(ids), np.concatenate(tin), wid); ids, tin, n, wid = [], [], 0, wid + 1
    tp, tc = threading.Thread(target=producer), threading.Thread(target=consumer)
    t0 = time.perf_counter(); tp.start(); tc.start(); tp.join(); tc.join()
    wall = time.perf_counter() - t0
    w = pd.DataFrame(wins); L = np.concatenate(lats)
    run = {"model": name, "window": W, "rate": rate or 0, "flows": len(idx_all), "wall_s": wall,
           "pipeline_flows_per_s": len(idx_all) / wall, "model_flows_per_s": len(idx_all) / w.predict_s.sum(),
           "predict_ms_median": 1000 * w.predict_s.median(),
           "lat_p50_s": np.percentile(L, 50), "lat_p95_s": np.percentile(L, 95), "lat_p99_s": np.percentile(L, 99)}
    return run, w, probs

def quality(probs, idx_all, name, thr):
    r = metrics.compute_metrics(y[idx_all], (probs[idx_all] >= thr).astype(int))
    return {"model": name, "thr": thr, **{k: r[k] for k in ("f1", "recall", "precision", "fpr")}}
print("sẵn sàng")

sẵn sàng


### Cell 3: Thí nghiệm A, throughput tối đa, toàn bộ Test (khoảng 5 phút)

In [5]:
REPEATS = 3
runs, wins_all, qual, probs_keep = [], [], [], {}
for W in (5000, 1000, 2000, 10000):
    for name in ("rf", "gnn"):
        for rep in range(REPEATS):
            run, w, pr = run_stream(name, W, None, order)
            run.update({"exp": "A_max_throughput", "rep": rep}); runs.append(run)
            w["run"] = len(runs) - 1; wins_all.append(w)
            if rep == 0:
                probs_keep[(name, W)] = pr
                for thr in ((0.5, 0.15) if name == "gnn" else (0.5,)):
                    qual.append({"window": W, **quality(pr, order, name, thr)})
        print("xong", W, name)
dfA = pd.DataFrame(runs)
print(dfA.groupby(["model", "window"])[["model_flows_per_s", "predict_ms_median"]].median().round(1))
print(pd.DataFrame(qual).round(4).to_string(index=False))

xong 5000 rf
xong 5000 gnn
xong 1000 rf
xong 1000 gnn
xong 2000 rf
xong 2000 gnn
xong 10000 rf
xong 10000 gnn
              model_flows_per_s  predict_ms_median
model window                                      
gnn   1000              84358.4               11.8
      2000             107522.1               18.6
      5000             116083.5               43.1
      10000            120581.1               82.4
rf    1000              30034.6               32.4
      2000              47567.8               40.7
      5000              74180.5               65.0
      10000             90410.9              105.7
 window model  thr     f1  recall  precision    fpr
   5000    rf 0.50 0.9896  0.9809     0.9984 0.0004
   5000   gnn 0.50 0.8971  0.8154     0.9971 0.0006
   5000   gnn 0.15 0.8937  0.8168     0.9866 0.0027
   1000    rf 0.50 0.9896  0.9809     0.9984 0.0004
   1000   gnn 0.50 0.8459  0.7398     0.9875 0.0023
   1000   gnn 0.15 0.8554  0.7813     0.9451 0.0111
   2000    rf 0.

### Cell 4: Thí nghiệm B, độ trễ ở tốc độ cố định (khoảng 8 phút)

In [6]:
N = 40_000
cs = np.concatenate([[0], np.cumsum(y[order])])
sub = order[int(np.argmax(cs[N:] - cs[:-N])):][:N]
for rate in (2000, 10000):
    for W in (1000, 2000, 5000):
        for name in ("rf", "gnn"):
            for rep in range(REPEATS):
                run, w, pr = run_stream(name, W, rate, sub)
                run.update({"exp": "B_latency", "rep": rep}); runs.append(run)
                w["run"] = len(runs) - 1; wins_all.append(w)
        print("xong", rate, W)
dfB = pd.DataFrame([r_ for r_ in runs if r_["exp"] == "B_latency"])
print(dfB.groupby(["rate", "window", "model"])[["lat_p50_s", "lat_p95_s", "predict_ms_median"]].median().round(3))

xong 2000 1000
xong 2000 2000
xong 2000 5000
xong 10000 1000
xong 10000 2000
xong 10000 5000
                    lat_p50_s  lat_p95_s  predict_ms_median
rate  window model                                         
2000  1000   gnn        0.238      0.462             11.053
             rf         0.260      0.484             32.939
      2000   gnn        0.496      0.925             17.730
             rf         0.518      0.951             41.296
      5000   gnn        1.268      2.392             39.930
             rf         1.299      2.421             70.220
10000 1000   gnn        0.058      0.102             11.031
             rf         0.080      0.123             33.077
      2000   gnn        0.115      0.201             17.491
             rf         0.139      0.228             42.314
      5000   gnn        0.288      0.512             40.010
             rf         0.318      0.542             68.592


### Cell 5: Xuất CSV

In [7]:
pd.DataFrame(runs).to_csv(f"{OUT}/streaming_runs.csv", index=False)
pd.concat(wins_all).to_csv(f"{OUT}/streaming_windows.csv", index=False)
pd.DataFrame(qual).to_csv(f"{OUT}/streaming_quality.csv", index=False)
pd.DataFrame({"flow_id": meta["flow_id"].values, "label_multi": meta["label_multi"].values, "y_true": y,
              "rf_prob": probs_keep[("rf", 5000)], "gnn_prob": probs_keep[("gnn", 5000)]}).iloc[order] \
  .to_csv(f"{OUT}/streaming_predictions_w5000.csv", index=False)
print(os.listdir(OUT))

['streaming_windows.csv', 'streaming_quality.csv', 'streaming_runs.csv', 'streaming_predictions_w5000.csv']
